# 13 · Training: speech enhancers (E3)

E3 asks what happens downstream when a predictor is used as a training reward.

1. `se_pretrain_s0..2` — a 2.6 M-parameter mask-based enhancer trained on VoiceBank-DEMAND (about 45 min).
2. `e3_grpo_<predictor>_s<seed>` — the pre-trained enhancer post-trained with GRPO for 2500 steps, using one predictor's
   score as the reward (R0, R3, R2 and the three baselines), about 20 min each. These start from the **released**
   pre-trained enhancer and reward predictor, so step 2 does not require step 1.

In [1]:
import os, sys
from pathlib import Path

# Works when started from the repo root or from notebooks/; needs the project environment
# (`source scripts/env.sh`, kernel "hackpatch" — see 00_setup.ipynb).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first key); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
PRETRAIN = ["se_pretrain_s0", "se_pretrain_s1", "se_pretrain_s2"]
KEYS = PRETRAIN + [k for k, v in reg.items() if v["kind"] == "e3_grpo"]
print(len(KEYS), "runs:", ", ".join(KEYS))

19 runs: se_pretrain_s0, se_pretrain_s1, se_pretrain_s2, e3_grpo_R0_s0, e3_grpo_R3_s0, e3_grpo_baseline_noise_s0, e3_grpo_baseline_pgd_s0, e3_grpo_baseline_random_s0, e3_grpo_R0_s1, e3_grpo_R3_s1, e3_grpo_baseline_noise_s1, e3_grpo_baseline_pgd_s1, e3_grpo_baseline_random_s1, e3_grpo_R0_s2, e3_grpo_R3_s2, e3_grpo_baseline_noise_s2, e3_grpo_baseline_pgd_s2, e3_grpo_baseline_random_s2, e3_grpo_R2_s0


In [4]:
# Resolved configuration of the first run (checkpoint paths already point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: se_pretrain
task: se_pretrain
phase: main
seed: 0
run_name: se_pretrain_s0__2026-09-15_16-10-03
model:
  n_fft: 512
  hop: 128
  hidden: 384
  rnn_hidden: 256
  rnn_layers: 2
  bidirectional: true
  conv_kernel: 3
  mask_max: 2.0
  init_log_std: -1.0
  min_log_std: -5.0
  max_log_std: 1.0
data:
  crop_seconds: 3.0
  num_workers: 8
  metric_workers: 8
  max_train_utts: null
  max_val_utts: null
  subset_seed: 0
pretrain:
  steps: 50000
  batch_size: 32
  lr: 0.001
  weight_decay: 1.0e-05
  warmup_steps: 1000
  cosine: true
  final_lr_frac: 0.02
  grad_clip: 5.0
  w_sisdr: 1.0
  w_mag: 10.0
  mag_compress: 0.3
  log_every: 50
  eval_every: 2500
  report_noisy: true
wandb:
  entity: alialavi
  mode: online
  group: se_pretrain



In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for se_pretrain_s0 ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_se_pretrain_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key se_pretrain_s0



## Retrained vs released GRPO runs: final validation record

In [6]:
rows = {}
released = repro.fetch_results()
for k in KEYS:
    new = repro.RUNS / "retrain" / k / "results"
    old = released / Path(reg[k]["run_dir"]).name / "results" if k in reg else None
    for src, d in (("released", old), ("retrained", new)):
        if d is not None and d.exists():
            csvs = sorted(d.glob("val_*.csv"))
            if csvs:
                rows[(k, src)] = pd.read_csv(csvs[-1]).mean(numeric_only=True)
pd.DataFrame(rows).T if rows else "no finished retrained runs yet"

,,dnsmos_bak,dnsmos_ovrl,dnsmos_p808,dnsmos_sig,energy_db,estoi,nisqa_mos,pesq,score_R0,score_R3,score_reward,sisdr,spk_sim,utmos,wer
e3_grpo_R0_s0,released,3.830044,3.072375,3.426144,3.452038,1.465451,0.686607,3.882851,2.088395,NaN,0.953385,3.548802,11.520205,0.901198,3.467354,0.157407
e3_grpo_R3_s0,released,3.908162,3.070605,3.441723,3.408023,-0.875986,0.712012,4.128842,2.155665,3.437240,NaN,3.545000,11.797032,0.918499,3.555753,0.135893
e3_grpo_baseline_noise_s0,released,3.849642,3.060961,3.498895,3.434607,-0.232466,0.691135,4.169956,2.126704,3.424948,1.303698,3.537083,10.452195,0.904709,3.485464,0.128644
e3_grpo_baseline_pgd_s0,released,3.851386,3.001294,3.385071,3.370286,-0.625565,0.684853,4.140488,2.190431,3.485521,1.228385,3.582865,9.614671,0.920083,3.482406,0.145866
e3_grpo_baseline_random_s0,released,4.035933,3.076598,3.450905,3.343723,-0.653899,0.701400,4.333592,2.211961,3.427500,2.852760,3.714375,11.160178,0.897296,3.493497,0.135535
e3_grpo_R0_s1,released,3.777304,3.032646,3.467810,3.438241,0.591571,0.683311,4.157364,2.188151,NaN,1.628854,3.650469,11.200481,0.896076,3.435455,0.124112
e3_grpo_R3_s1,released,4.050483,3.170043,3.367769,3.444095,-1.203335,0.702673,4.017073,2.223464,3.477448,NaN,3.382292,10.259416,0.913501,3.456609,0.141009
e3_grpo_baseline_noise_s1,released,3.845516,3.039331,3.414489,3.411109,0.517682,0.692962,4.325659,2.174732,3.612344,2.353281,3.559896,10.265266,0.890656,3.465990,0.119915
e3_grpo_baseline_pgd_s1,released,3.893725,3.026175,3.464668,3.370592,0.993354,0.697617,4.289531,2.239945,3.590625,2.407135,3.538646,10.683669,0.899377,3.502920,0.124366
e3_grpo_baseline_random_s1,released,3.940390,3.070702,3.512508,3.397610,-0.302232,0.701587,4.431805,2.191924,3.598750,2.427969,3.667813,11.365308,0.888918,3.504797,0.127747
